# Sequence Visualization

This notebook loads a Qupyt sequence YAML file from the ".qupyt/sequences/" folder and visualizes pulses per channel with block timing.

It is organized into four parts:
1. imports
2. data loading
3. plotting
4. optional export

In [15]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
import plotly.io as pio
import yaml

pio.templates.default = "plotly_white"

## Data Loading

Helpers for locating and parsing sequence YAML files.

In [16]:
def get_sequence_path(ps_step: int) -> Path:
    """Return `~/.qupyt/sequences/sequence_<ps_step>.yaml`."""
    return Path.home() / ".qupyt" / "sequences" / f"sequence_{ps_step}.yaml"


def _to_block_name(value) -> str:
    """Normalize numeric block identifiers to `block_<n>` format."""
    if isinstance(value, int):
        return f"block_{value}"
    if isinstance(value, str) and value.isdigit():
        return f"block_{value}"
    return str(value)


def _read_sequence_yaml(yaml_path: str | Path) -> dict:
    """Read and validate a sequence YAML file."""
    path = Path(yaml_path).expanduser()
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle) or {}

    if not isinstance(data, dict):
        raise ValueError(f"Unexpected YAML structure in {path}")

    return data


def load_blocks(yaml_path: str | Path, expand_repeats: bool = False):
    """Load pulse rows and block timing from a Qupyt sequence YAML.

    Parameters
    ----------
    yaml_path:
        Path to the YAML sequence file.
    expand_repeats:
        If True, each block repeat becomes its own block instance with repeated pulses.
        If False, pulses are drawn once per sequencing entry while each block still occupies
        `repeat * total_duration` on the timeline.

    Returns
    -------
    pulses_df, blocks_df, block_duration
        `pulses_df` contains pulse geometry and metadata.
        `blocks_df` contains timeline spans for each sequencing entry.
        `block_duration` is the YAML `total_duration` (microseconds).
    """
    data = _read_sequence_yaml(yaml_path)

    order = data.get("sequencing_order")
    repeats = data.get("sequencing_repeats")

    if order is None:
        order = sorted(
            [k for k, v in data.items() if isinstance(v, dict)],
            key=lambda name: int(re.search(r"[0-9]+", name).group()) if re.search(r"[0-9]+", name) else name,
        )

    order = [_to_block_name(item) for item in order]

    if repeats is None or len(repeats) != len(order):
        repeats = [1] * len(order)
    repeats = [int(rep) for rep in repeats]

    block_duration = float(data.get("total_duration", 0.0))
    if block_duration <= 0:
        raise ValueError("Missing or invalid `total_duration` in YAML.")

    if expand_repeats:
        sequence = []
        sequence_repeats = []
        for block, rep in zip(order, repeats):
            sequence.extend([block] * rep)
            sequence_repeats.extend([1] * rep)
    else:
        sequence = list(order)
        sequence_repeats = list(repeats)

    pulse_rows = []
    block_rows = []
    timeline_t = 0.0

    for idx, (block, rep) in enumerate(zip(sequence, sequence_repeats)):
        block_data = data.get(block, {})
        local_rows = []

        for channel, pulses in block_data.items():
            if not isinstance(pulses, dict):
                continue

            for pulse_name, pulse in pulses.items():
                if not isinstance(pulse, dict):
                    continue

                start = pulse.get("start")
                duration = pulse.get("duration")
                if start is None or duration is None:
                    continue

                local_rows.append(
                    {
                        "instance": idx,
                        "block": block,
                        "repeat": int(rep),
                        "channel": channel,
                        "pulse": pulse_name,
                        "local_start": float(start),
                        "duration": float(duration),
                        "amplitude": float(pulse.get("amplitude", 1.0)),
                        "frequency": float(pulse.get("frequency", 1.0)),
                        "phase": pulse.get("phase"),
                    }
                )

        for row in local_rows:
            row["global_start"] = timeline_t + row["local_start"]
            row["global_end"] = row["global_start"] + row["duration"]
            pulse_rows.append(row)

        span = block_duration if expand_repeats else block_duration * int(rep)
        block_rows.append(
            {
                "idx": idx,
                "block": block,
                "repeat": int(rep),
                "start": timeline_t,
                "end": timeline_t + span,
            }
        )
        timeline_t += span

    pulses_df = pd.DataFrame(pulse_rows)
    blocks_df = pd.DataFrame(block_rows)

    if pulses_df.empty:
        print("No pulses found in sequence file.")
    else:
        print(
            f"Loaded {len(pulses_df)} pulses across "
            f"{pulses_df['channel'].nunique()} channels and {len(blocks_df)} blocks."
        )

    return pulses_df, blocks_df, block_duration

## Plotting

Functions for rendering pulses and adding a block lane below channels.

In [17]:
def plot_sequence(
    pulses_df: pd.DataFrame,
    blocks_df: pd.DataFrame,
    y_gap: float = 0.7,
    add_block_dividers: bool = True,
    shade_blocks: bool = True,
    center_pulses: bool = True,
    min_pulse_height: float = 0.02,
    amp_scale: float = 1.0,
    alpha: float = 0.7,
    block_duration: float | None = None,
):
    """Create an interactive pulse plot with hover details."""
    if pulses_df.empty or blocks_df.empty:
        print("Nothing to plot.")
        return None

    fixed_colors = {
        "LASER": f"rgba(50,205,50,{alpha})",
        "READ": f"rgba(238,44,44,{alpha})",
        "MW": f"rgba(0,101,189,{alpha})",
        "START": f"rgba(0,0,0,{alpha})",
    }
    extra_colors = [
        f"rgba(255,140,0,{alpha})",
        f"rgba(255,215,0,{alpha})",
        f"rgba(174,20,174,{alpha})",
    ]

    def norm_channel(name: str) -> str:
        return str(name).strip().upper()

    channels_raw = list(pulses_df["channel"].unique())
    channel_norm = {channel: norm_channel(channel) for channel in channels_raw}

    priority = ["START", "LASER", "READ", "MW"]
    priority_present = []
    for preferred in priority:
        for channel in channels_raw:
            if channel_norm[channel] == preferred and channel not in priority_present:
                priority_present.append(channel)

    channels = priority_present + [c for c in channels_raw if c not in priority_present]

    palette = pc.qualitative.Set2
    channel_colors = {}
    color_idx = 0
    for channel in channels:
        normalized = channel_norm[channel]
        if normalized in fixed_colors:
            channel_colors[channel] = fixed_colors[normalized]
            continue

        if color_idx < len(extra_colors):
            channel_colors[channel] = extra_colors[color_idx]
        elif color_idx < len(extra_colors) + len(palette):
            channel_colors[channel] = palette[color_idx - len(extra_colors)]
        else:
            rgb = pc.unlabel_rgb(pc.sample_colorscale("Turbo", (color_idx * 0.13) % 1.0)[0])
            channel_colors[channel] = f"rgb({int(rgb[0])},{int(rgb[1])},{int(rgb[2])})"
        color_idx += 1

    traces = []
    ytickvals, yticktext = [], []
    n_channels = len(channels)

    for channel_idx, channel in enumerate(channels):
        y_center = (n_channels - 1 - channel_idx) * y_gap
        ytickvals.append(y_center)
        yticktext.append(channel)

        channel_rows = pulses_df[pulses_df["channel"] == channel]
        color = channel_colors[channel]

        for _, row in channel_rows.iterrows():
            t0, t1 = float(row["global_start"]), float(row["global_end"])
            height = max(abs(float(row["amplitude"])) * amp_scale, min_pulse_height)
            half_height = 0.5 * height

            if center_pulses:
                y0, y1 = y_center - half_height, y_center + half_height
            else:
                y0, y1 = y_center, y_center + height

            traces.append(
                go.Scatter(
                    x=[t0, t1, t1, t0, t0],
                    y=[y0, y0, y1, y1, y0],
                    mode="lines",
                    line=dict(color=color, width=1),
                    fill="toself",
                    fillcolor=color,
                    hoverinfo="skip",
                    showlegend=False,
                )
            )

    marker_x, marker_y, marker_text, marker_color = [], [], [], []
    for channel_idx, channel in enumerate(channels):
        y_center = (n_channels - 1 - channel_idx) * y_gap
        channel_rows = pulses_df[pulses_df["channel"] == channel]
        color = channel_colors[channel]

        for _, row in channel_rows.iterrows():
            t0, t1 = float(row["global_start"]), float(row["global_end"])
            cx = 0.5 * (t0 + t1)

            repeat_text = ""
            if pd.notna(row.get("repeat")) and int(row["repeat"]) > 1:
                repeat_text = f" x{int(row['repeat'])}"

            hover = (
                f"block: {row['block']} (#{row['instance']}){repeat_text}"
                f"<br>channel: {channel}"
                f"<br>pulse: {row['pulse']}"
                f"<br>t0={t0:g} -> t1={t1:g}"
                f"<br>duration={row['duration']:g}"
                f"<br>amplitude={row['amplitude']:g}"
            )
            if pd.notna(row.get("frequency")):
                hover += f"<br>frequency={row['frequency']:g}"
            if pd.notna(row.get("phase")):
                hover += f"<br>phase={row['phase']:g}"

            marker_x.append(cx)
            marker_y.append(y_center)
            marker_text.append(hover)
            marker_color.append(color)

    traces.append(
        go.Scatter(
            x=marker_x,
            y=marker_y,
            mode="markers",
            marker=dict(size=12, opacity=0.01, color=marker_color),
            hovertemplate="%{text}<extra></extra>",
            text=marker_text,
            showlegend=False,
        )
    )

    t_min = float(pulses_df["global_start"].min())
    t_max = float(pulses_df["global_end"].max())
    span = max(t_max - t_min, 1e-9)
    pad = 0.02 * span

    title = "Pulse Sequence"
    if block_duration is not None and float(block_duration) > 0:
        title = f"{title} (block={float(block_duration):g} us)"

    fig = go.Figure(traces)
    fig.update_layout(
        title=title,
        xaxis=dict(
            title="Time [us]",
            range=[t_min - pad, t_max + pad],
            rangeslider=dict(visible=True),
            zeroline=False,
        ),
        yaxis=dict(
            title="Channels",
            tickmode="array",
            tickvals=ytickvals,
            ticktext=yticktext,
            zeroline=False,
        ),
        dragmode="zoom",
        hovermode="closest",
        height=max(450, int(len(channels) * 120 * y_gap)),
        margin=dict(l=70, r=20, t=50, b=40),
    )

    if shade_blocks:
        for _, block in blocks_df.iterrows():
            if int(block["idx"]) % 2 == 0:
                fig.add_vrect(
                    x0=float(block["start"]),
                    x1=float(block["end"]),
                    fillcolor="rgba(0,0,0,0.03)",
                    layer="below",
                    line_width=0,
                )

    if add_block_dividers and len(blocks_df) > 1:
        for _, block in blocks_df.iloc[1:].iterrows():
            fig.add_vline(
                x=float(block["start"]),
                line_dash="dot",
                line_color="gray",
                opacity=0.5,
            )

    return fig


def add_blocks_lane_to_sequence_figure(
    fig: go.Figure,
    blocks_df: pd.DataFrame,
    lane_height: float = 0.55,
    lane_gap: float = 0.25,
    show_repeat: bool = True,
    show_block_name: bool = True,
    shade_alternate: bool = True,
    *,
    amp_scale: float | None = None,
    scale_lane_with_amp: bool = True,
):
    """Add a block-schedule lane underneath an existing sequence figure."""
    yaxis = fig.layout.yaxis
    current_range = getattr(yaxis, "range", None)

    if current_range is None:
        y_min, y_max = float("inf"), float("-inf")
        for trace in fig.data:
            if hasattr(trace, "y") and trace.y is not None:
                values = [value for value in trace.y if value is not None]
                if values:
                    y_min = min(y_min, min(values))
                    y_max = max(y_max, max(values))
        if y_min == float("inf"):
            y_min, y_max = 0.0, 1.0
    else:
        y_min, y_max = current_range

    if amp_scale is not None and scale_lane_with_amp:
        lane_height_eff = lane_height * float(amp_scale)
        lane_gap_eff = lane_gap * float(amp_scale)
    else:
        lane_height_eff = lane_height
        lane_gap_eff = lane_gap

    lane_top = y_min - lane_gap_eff
    lane_bottom = lane_top - lane_height_eff
    fig.update_yaxes(range=[lane_bottom - 0.1 * lane_height_eff, y_max])

    for idx, block in blocks_df.iterrows():
        x0 = float(block["start"])
        x1 = float(block["end"])
        block_name = str(block.get("block", ""))
        fillcolor = "rgba(0,0,0,0.06)" if (shade_alternate and idx % 2 == 0) else "rgba(0,0,0,0.02)"

        fig.add_shape(
            type="rect",
            x0=x0,
            x1=x1,
            y0=lane_bottom,
            y1=lane_top,
            xref="x",
            yref="y",
            line=dict(color="black", width=2),
            fillcolor=fillcolor,
            layer="below",
        )

        if show_block_name:
            fig.add_annotation(
                x=0.5 * (x0 + x1),
                y=0.5 * (lane_bottom + lane_top),
                xref="x",
                yref="y",
                text=block_name,
                showarrow=False,
                font=dict(size=11),
            )

        if show_repeat and "repeat" in blocks_df.columns and pd.notna(block.get("repeat")):
            fig.add_annotation(
                x=0.5 * (x0 + x1),
                y=lane_top,
                xref="x",
                yref="y",
                text=f"x{int(block['repeat'])}",
                showarrow=False,
                yshift=10,
                font=dict(size=11),
            )

    return fig

## Export

Helper to save the generated Plotly figure.

In [18]:
def save_plotly_figure(
    fig: go.Figure,
    name: str,
    out_dir: str | Path = "ps_plots",
    *,
    html: bool = True,
    json_export: bool = False,
    include_plotlyjs: str | bool = "cdn",
    auto_open: bool = False,
    add_timestamp: bool = True,
    timestamp_fmt: str = "%Y%m%d_%H%M%S",
) -> dict[str, Path]:
    """Save a Plotly figure and return exported paths."""
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    if add_timestamp:
        name = f"{name}_{datetime.now().strftime(timestamp_fmt)}"

    safe_name = "".join(ch if ch.isalnum() or ch in ("-", "_", ".") else "_" for ch in name).strip("_")
    exported = {}

    if html:
        html_path = out_path / f"{safe_name}.html"
        fig.write_html(
            html_path,
            include_plotlyjs=include_plotlyjs,
            full_html=True,
            auto_open=auto_open,
        )
        exported["html"] = html_path

    if json_export:
        json_path = out_path / f"{safe_name}.json"
        pio.write_json(fig, json_path, pretty=True)
        exported["json"] = json_path

    return exported

## Render Sequence

Set `ps_step` to choose which sequence file to visualize.

In [19]:
ps_step = 0
yaml_path = get_sequence_path(ps_step)

pulses_df, blocks_df, block_duration = load_blocks(yaml_path, expand_repeats=False)

fig = plot_sequence(
    pulses_df,
    blocks_df,
    y_gap=0.4,
    amp_scale=0.3,
    alpha=0.8,
    block_duration=block_duration,
)

if fig is not None:
    fig = add_blocks_lane_to_sequence_figure(
        fig,
        blocks_df[["start", "end", "block", "repeat"]],
        amp_scale=0.6,
    )
    fig.show()

Loaded 19 pulses across 4 channels and 5 blocks.


## Save Figure (Optional)

Writes output to `ps_plots/`.

In [ ]:
if fig is not None:
    saved_paths = save_plotly_figure(
        fig,
        name="pulse_sequence",
        out_dir="ps_plots",
        include_plotlyjs=True,
    )
    saved_paths